#### Semantic Search for Grocery

We are using below things:

Database: ChromaDB
Embedding Model: sentence-transformers/all-MiniLM-L6-v2  from hugging face

##### 1. Loading Libraries and Dataset

In [42]:
import chromadb
from sentence_transformers import SentenceTransformer
import os
import json
from chromadb.utils import embedding_functions

pre-loaded db is already there ./chroma_store/chroma.sqlite3

In [39]:
grocery_data = None
with open("./grocery_data.json", "rb") as rJ:

    grocery_data = json.load(rJ)

len(grocery_data)

27439

##### 2. Database Setup and Configuration

In [41]:
# path to store the database
chroma_db_persist_dir = os.path.join(os.getcwd(), 'chroma_store')

# creating client with persistent dir
chroma_persistent_client = chromadb.PersistentClient(path=chroma_db_persist_dir)

In [ ]:
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

# setting up embedding function

st_ef = embedding_functions.SentenceTransformerEmbeddingFunction(embedding_model_name)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6868.58it/s]


In [ ]:
# sample code to generate the embedding using above model
# sentences = ["This is an example sentence", "Each sentence is converted"]

# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
# embeddings = model.encode(sentences)
# print(embeddings)

now we have set the embedding function so we do not need to do it manually, chromadb will do for us

Let's create the collection in db

In [46]:
# creating collection

collection_name = "grocery_collection"

# re-executing script prevention and purging as well
if collection_name in [item.name for item in chroma_persistent_client.list_collections()]:
  chroma_persistent_client.delete_collection(name=collection_name)

# creating the collection with embedding function 
grocery_collection = chroma_persistent_client.create_collection(
  name=collection_name,
  embedding_function=st_ef)

print("document_count:", grocery_collection.count())


document_count: 0


##### 3. Feeding data into database

Now to add data into chromDB, we need lists; documents, ids, metadatas, embeddings (in case external) in an order. So let's prepare the data

In [47]:
grocery_data[0]

{'product': 'Garlic Oil - Vegetarian Capsule 500 mg',
 'category': 'Beauty & Hygiene',
 'sub_category': 'Hair Care',
 'brand': 'Sri Sri Ayurveda ',
 'type': 'Hair Oil & Serum',
 'description': 'This Product contains Garlic Oil that is known to help proper digestion, maintain proper cholesterol levels, support cardiovascular and also build immunity.  For Beauty tips, tricks & more visit https://bigbasket.blog/'}

In [48]:
import uuid
ids = []
documents = []
metadatas = []

for item in grocery_data:
    ids.append(str(uuid.uuid4()))
    documents.append(f"""Name: {item["product"]}\nCategory: {item["category"]}\nSub Category: {item["sub_category"]}\nProduct Type: {item["type"]}\nProduct Description: {item["description"]}""")
    metadatas.append({
        'category': item['category'],
        'sub_category': item['sub_category'],
        'brand': item['brand'],
        'type':  item['type'],
    })

print("length of ids: ", len(ids))
print("length of documents: ", len(documents))
print("length of metadatas: ", len(metadatas))

length of ids:  27439
length of documents:  27439
length of metadatas:  27439


Let's feed into collection as chunk

In [52]:
# as data is huge and we have limit in SQLite so we will be doing insertion in chunk

print("pre insertion grocery_collection document count: ", grocery_collection.count())
batch_size = 4000
for i in range(0, len(documents), batch_size):

    _ids=ids[i: i + batch_size]
    _documents=documents[i: i + batch_size]
    _metadatas=metadatas[i: i + batch_size]

    grocery_collection.add(
        ids=_ids,
        documents=_documents,
        metadatas=_metadatas
    )   


 
print("post insertion grocery_collection document count: ", grocery_collection.count())

pre insertion grocery_collection document count:  0
post insertion grocery_collection document count:  27439


##### 4. Let's Query

In [53]:
result = grocery_collection.query(
    query_texts = ["almonds"],
    n_results = 5
)
result

{'ids': [['2bd9a0da-cb84-4b37-b79c-63b7778e8ede',
   '43f092f9-dd18-4192-8a0b-2d9fdc7d9de6',
   '82cd3a6d-6ee3-4bda-99b8-f2e23cd53f21',
   '87993459-2c6d-4808-baa6-fc89dfa56242',
   'f229db40-8fd8-4fb3-ae5e-cab53f58e1b4']],
 'embeddings': None,
 'documents': [["Name: Almonds - Roasted & Salted\nCategory: Foodgrains, Oil & Masala\nSub Category: Dry Fruits\nProduct Type: Almonds\nProduct Description: California Almonds are bite-sized all-rounders when it comes to keeping you young & fit.\nIt's food for the brain, keeping your memory sharp. They are packed full of nutrients.\nThe wholesome constituents of these nuts provide the required amount of energy to the body and help maintain brain health.\nMakes a healthy and tasty addition to both sweets and savouries.\nStore in a cool, dry place in an airtight container and preferably refrigerate after opening\n\nClick here for unique and delicious recipes - https://www.bigbasket.com/flavors/collections/231/dry-fruits-berries-nuts/",
   'Name: A

In [ ]:
result = grocery_collection.query(
    query_texts = ["healthy snacks which is low in fat"],
    n_results = 5
)
result

{'ids': [['7dea6590-0d24-4ac6-90ea-5415540db0ec',
   'dd2a7dbd-15ae-491e-a2bc-a06f18c568b8',
   '7ea8d844-effe-41b0-b516-49319e844d14',
   '8e487c93-f5b2-47ce-ad3f-71e6b010a681',
   '1864f4ad-89bf-4aa0-aee3-f04185617081']],
 'embeddings': None,
 'documents': [['Name: Healthy Bites - Mini Health Bars\nCategory: Gourmet & World Food\nSub Category: Snacks, Dry Fruits, Nuts\nProduct Type: Healthy, Baked Snacks\nProduct Description: Mini Health Bars are one of the most nutritious snacks. It has got properties of different grains and nuts. These bars are a superfood for your body and brains, made with good quality ingredients. A balance of good taste and healthy munching is on your way to being added to your favourites.\xa0\n‘Snack Amor’ has launched healthy options to keep your fitness in check. The brand has come up with premium quality snacks, nuts and dried fruits collection e.t.c. Loaded with nutrients and enriching qualities these super snacks are a must-have.',
   'Name: Snacks - Roas